# Tool Use from Scratch

**What you will build:** a real agent — the full ReAct loop — in about forty lines of Python, using nothing but the `openai` SDK. By the end you'll have written by hand what PydanticAI and LangGraph do for you, and you'll understand every line of it. This is the most important notebook in Block 0.

> **Run it:** open in [Google Colab](https://colab.research.google.com/github/ezponda/ai-agents-course/blob/main/courses/python_code/book/03_tool_use_from_scratch.ipynb) (nothing to install), or run locally in Jupyter. Each notebook installs what it needs in its first cell.

In [ ]:
# Setup — same as notebook 0.1. Run this once per notebook.
!pip install -q openai pydantic

import os, getpass, json
from openai import OpenAI

if not os.environ.get("OPENROUTER_API_KEY"):
    os.environ["OPENROUTER_API_KEY"] = getpass.getpass("Paste your OpenRouter API key: ")

MODEL = "meta-llama/llama-3.3-70b-instruct"   # change to any model on openrouter.ai/models
client = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=os.environ["OPENROUTER_API_KEY"])
print("Ready. Model:", MODEL)

## Recap: tool calling is a protocol

From notebook 0.0: the model can't run anything itself. It only produces text — including text that *requests* a tool. The hand-off has four moves, and we're about to implement all four:

```
  1. You describe the tools in the request      (name, description, arguments schema)
  2. The model replies with a structured call:  get_weather(city="Bilbao")
  3. YOUR code runs the real function:          -> "18C, rain"
  4. You send the result back; the model loops or answers
```

## Step 1 — Write the tools as plain Python functions

Tools are just functions. Nothing special about them yet.

In [ ]:
def get_weather(city: str) -> str:
    """Return a (fake) current weather report for a city."""
    fake = {"bilbao": "18C, light rain", "madrid": "31C, sunny", "london": "14C, cloudy"}
    return fake.get(city.lower(), "20C, clear")

import ast, operator
_OPS = {ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul,
        ast.Div: operator.truediv, ast.USub: operator.neg, ast.UAdd: operator.pos}
def safe_eval(expr):
    """Evaluate +,-,*,/ and parentheses only. No eval(): '9**9**9' can't freeze the kernel, and 'import ...' can't run."""
    def ev(n):
        if isinstance(n, ast.Expression): return ev(n.body)
        if isinstance(n, ast.Constant) and isinstance(n.value, (int, float)): return n.value
        if isinstance(n, ast.BinOp) and type(n.op) in _OPS: return _OPS[type(n.op)](ev(n.left), ev(n.right))
        if isinstance(n, ast.UnaryOp) and type(n.op) in _OPS: return _OPS[type(n.op)](ev(n.operand))
        raise ValueError('unsupported expression')
    return ev(ast.parse(expr, mode='eval'))

def calculator(expression: str) -> str:
    """Evaluate a basic arithmetic expression like '12 * (3 + 4)'."""
    try:
        return str(safe_eval(expression))
    except ZeroDivisionError:
        return "Error: division by zero."
    except (ValueError, SyntaxError):
        return "Error: only +, -, *, / and parentheses are supported."

# A lookup so the loop can call a tool by name:
TOOL_FUNCTIONS = {"get_weather": get_weather, "calculator": calculator}

## Step 2 — Describe the tools for the model

The model needs to know each tool's **name**, **what it does**, and **what arguments it takes**. That description is a JSON schema. The `description` text is not decoration — it's how the model decides *when* to use the tool. (Block 1.2 is entirely about writing good tool descriptions; for now, be clear and literal.)

In [ ]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "Get the current weather for a given city.",
            "parameters": {
                "type": "object",
                "properties": {"city": {"type": "string", "description": "City name, e.g. Bilbao"}},
                "required": ["city"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "calculator",
            "description": "Evaluate a basic arithmetic expression like '12 * (3 + 4)'.",
            "parameters": {
                "type": "object",
                "properties": {"expression": {"type": "string", "description": "The arithmetic expression"}},
                "required": ["expression"],
            },
        },
    },
]

## Step 3 — One turn, by hand

Let's do a single round-trip so you can see the raw signal. Ask something that needs a tool, pass `tools=`, and inspect what comes back.

In [ ]:
messages = [{"role": "user", "content": "What's the weather in Bilbao right now?"}]

resp = client.chat.completions.create(model=MODEL, messages=messages, tools=tools, temperature=0)
choice = resp.choices[0]

print("finish_reason:", choice.finish_reason)          # -> 'tool_calls' (not 'stop')
print("content:      ", choice.message.content)         # usually None when calling a tool
print("tool_calls:   ", choice.message.tool_calls)      # the structured request

Read that output carefully. Instead of answering, the model set `finish_reason='tool_calls'` and returned a `tool_calls` list. Each entry has an `id`, a function `name`, and `arguments` **as a JSON string** you must parse. That's move 2 of the protocol. Now we run the tool (move 3) and feed the result back (move 4).

In [ ]:
tool_call = choice.message.tool_calls[0]
name = tool_call.function.name
args = json.loads(tool_call.function.arguments)     # arguments come as a JSON string
print("model asked to call:", name, "with", args)

result = TOOL_FUNCTIONS[name](**args)               # MOVE 3: your code runs it
print("tool result:", result)

# MOVE 4: append the assistant's tool request AND the tool result, then call again.
messages.append(choice.message)                     # the assistant message that requested the tool
messages.append({"role": "tool", "tool_call_id": tool_call.id, "content": result})

final = client.chat.completions.create(model=MODEL, messages=messages, tools=tools, temperature=0)
print("\nfinal answer:", final.choices[0].message.content)

That is the entire mechanic. The only thing left is to wrap it in a `while` loop so the model can chain several tools (and so it works for questions that need zero tools, or five).

## Step 4 — The agent loop

Here it is: the complete ReAct loop. Read it against the four-move protocol and the THINK/ACT/OBSERVE diagram from 0.0 — they line up exactly.

In [ ]:
def run_agent(user_message, system="You are a helpful assistant. Use tools when they help.", max_turns=10):
    messages = [{"role": "system", "content": system},
                {"role": "user", "content": user_message}]

    total_tokens = 0
    for _ in range(max_turns):
        resp = client.chat.completions.create(
            model=MODEL, messages=messages, tools=tools, temperature=0,
        )
        total_tokens += resp.usage.total_tokens               # tally the cost across the whole loop
        msg = resp.choices[0].message
        messages.append(msg)                                  # remember what the model said

        if not msg.tool_calls:                                # OBSERVE: no tool wanted -> done
            print(f"  [cost] {total_tokens} tokens across this run")
            return msg.content

        for tc in msg.tool_calls:                             # ACT: run every requested tool
            try:                                              # parse AND run inside the try: both can fail,
                args = json.loads(tc.function.arguments)      #   and the model can emit malformed JSON on weaker routes
                result = TOOL_FUNCTIONS[tc.function.name](**args)
            except Exception as e:
                result = f"Error: {e}"                         # feed the error back; the model can react to it
            print(f"  [tool] {tc.function.name}({args}) -> {result}")
            messages.append({"role": "tool", "tool_call_id": tc.id, "content": str(result)})

    return "Stopped: reached max turns."                       # the safety valve

```{note}
Two production habits are baked into this loop. Tool calls are wrapped in **`try/except`** — a real tool (a web API, a database) *will* fail sometimes, and the agent should see the error as just another observation and recover, not crash. And we tally **`usage.total_tokens`** so you can feel the cost: an agent makes several model calls per task, and you pay for each one. (The `calculator` here uses the AST-based `safe_eval` from the tools cell — no `eval()`, so `9**9**9` can't freeze your kernel.)
```

Try it on a question that needs **two** tools in sequence:

In [ ]:
answer = run_agent("What's the weather in Madrid, and what is 231 * 17?")
print("\nAgent:", answer)

Watch the `[tool]` lines: the model decided, on its own, to call `get_weather` and `calculator`, then combined the results into one answer. You didn't write any `if` about weather or math — the model routed itself. **That's the jump from workflow to agent**, and you now own every line of it.

::::{dropdown} 🔍 How the loop actually runs. See the message list grow
:color: info

For the two-tool question, the `messages` list evolves like this — each turn appends more, and *that growing list is the agent's entire memory of the task*:

```
turn 1  [system, user]                                  -> model returns 2 tool_calls
        append assistant(tool_calls)
        append tool(get_weather -> "31C, sunny")
        append tool(calculator  -> "3927")
turn 2  [system, user, assistant, tool, tool]           -> model returns final text
        no tool_calls  ->  return the answer
```

There is no hidden state. Everything the model knows about the task is in that list, which is why context management (0.6) matters so much once tasks get long.
::::

::::{dropdown} 🛠️ Try it yourself
:color: secondary

1. **Add a tool.** Write `word_count(text: str) -> int`, add it to `TOOL_FUNCTIONS` and to the `tools` schema, then ask "How many words are in this sentence?". Three edits, no loop changes.
2. **Watch the description matter.** Make the `calculator` description vague (`"does stuff with numbers"`) and ask a math question. The model may stop choosing it. This previews Block 1.2 — descriptions are the steering wheel.
3. **Trigger the safety valve.** Lower `max_turns` to `1` and ask the two-tool question. See it stop cleanly instead of answering. A missing stop condition is how real agents hang.
::::

::::{dropdown} 🔧 Common issues (and fixes)
:color: secondary

| Symptom | Likely cause | Fix |
|---------|--------------|-----|
| **The agent never calls a tool** | Vague tool `description`, or a model that doesn't support tool-calling | Sharpen the description (say *when* to use it); switch `MODEL` to a tool-capable one |
| **It loops until `max_turns`** | No clear terminal result, so the model keeps trying | Keep the `max_turns` guard; make tools return unambiguous results |
| **A tool raises and the run dies** | Unhandled exception in tool code | Wrap tool calls in `try/except` and feed the error back (as `run_agent` does) |
| **`json.loads(arguments)` fails** | Model produced malformed arguments (model-dependent) | Catch it and return an error string so the model can retry |
::::

**What's next:** you've built an agent where the *model* drives. In **0.4** we go the other way — **workflows**, where *you* drive — so you can tell which one a task actually needs.